In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Generar datos
X, y = make_moons(n_samples=500, noise=0.2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Escalar
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Visualizar
plt.scatter(X_train_scaled[:, 0], X_train_scaled[:, 1], c=y_train, cmap='bwr', edgecolor='k', alpha=0.7)
plt.title("Two Moons - Dataset de Entrenamiento")
plt.xlabel("Feature 1"); plt.ylabel("Feature 2")
plt.show()


In [ ]:
from sklearn.svm import SVC

kernels = {
    'linear': 'Kernel lineal: separa con un hiperplano',
    'poly':   'Kernel polinomial (grado 3): permite curvas polinómicas',
    'rbf':    'Kernel gaussiano (RBF): mide similitud exponencial',
    'sigmoid':'Kernel sigmoide: similar a una red neuronal simple'
}

models = {}
for k in kernels:
    if k == 'poly':
        svm = SVC(kernel=k, degree=3, C=1.0, gamma='scale', random_state=42)
    else:
        svm = SVC(kernel=k, C=1.0, gamma='scale', random_state=42)
    svm.fit(X_train_scaled, y_train)
    models[k] = svm
    print(f"{k}: train acc = {svm.score(X_train_scaled, y_train):.2f}, "
          f"test acc = {svm.score(X_test_scaled, y_test):.2f}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_boundary(model, X, y, ax, title):
    xx, yy = np.meshgrid(
        np.linspace(X[:,0].min()-1, X[:,0].max()+1, 300),
        np.linspace(X[:,1].min()-1, X[:,1].max()+1, 300)
    )
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.2, cmap='bwr')
    ax.scatter(X[:,0], X[:,1], c=y, cmap='bwr', edgecolor='k', alpha=0.6)
    ax.set_title(title)
    ax.set_xlabel("Feat 1"); ax.set_ylabel("Feat 2")

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for ax, (k, svm) in zip(axes.flatten(), models.items()):
    plot_boundary(svm, X_test_scaled, y_test, ax, f"{k} kernel")
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Generar dataset con ruido y solapamiento
X, y = make_classification(
    n_samples=600,
    n_features=4,
    n_informative=3,
    n_redundant=1,
    n_clusters_per_class=2,
    class_sep=1.0,
    flip_y=0.1,
    random_state=0
)

# División train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0
)

# Escalado de características
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)


In [ ]:
from sklearn.svm import SVC

kernels = ['linear', 'poly', 'rbf', 'sigmoid']
models = {}

for kernel in kernels:
    if kernel == 'poly':
        model = SVC(kernel=kernel, degree=3, C=1.0, gamma='scale',
                    probability=True, random_state=0)
    else:
        model = SVC(kernel=kernel, C=1.0, gamma='scale',
                    probability=True, random_state=0)
    model.fit(X_train_scaled, y_train)
    models[kernel] = model
    print(f"{kernel:7s} | Train acc: {model.score(X_train_scaled, y_train):.2f} | "
          f"Test acc: {model.score(X_test_scaled, y_test):.2f}")


In [ ]:
from sklearn.decomposition import PCA

# PCA para visualizar en 2D
pca = PCA(n_components=2, random_state=0)
X_train_2d = pca.fit_transform(X_train_scaled)

# Crear malla en espacio PCA
x_min, x_max = X_train_2d[:,0].min()-1, X_train_2d[:,0].max()+1
y_min, y_max = X_train_2d[:,1].min()-1, X_train_2d[:,1].max()+1
xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 300),
    np.linspace(y_min, y_max, 300)
)

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for ax, (kernel, model) in zip(axes, models.items()):
    # Invertir PCA para evaluar en espacio original
    grid_pca = np.c_[xx.ravel(), yy.ravel()]
    grid_orig = pca.inverse_transform(grid_pca)
    Z = model.predict(grid_orig).reshape(xx.shape)

    # Plot de la frontera de decisión
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='bwr')
    # Puntos de entrenamiento
    ax.scatter(X_train_2d[:,0], X_train_2d[:,1], c=y_train,
               cmap='bwr', edgecolor='k', alpha=0.6)
    # Vectores de soporte
    sv = model.support_
    sv_pca = X_train_2d[sv]
    ax.scatter(sv_pca[:,0], sv_pca[:,1],
               facecolors='none', edgecolors='k', s=100, label='Soportes')

    ax.set_title(f"SVM ({kernel} kernel)")
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
    ax.legend()

plt.tight_layout()
plt.show()
